# Diamond Price Regression

## 1. Objective

Predict diamond **price** from carat, cut, color, clarity, depth, table, and x/y/z.
Price is the target and never enters the model as a feature. Clarity is a known input.

Three representations answer three questions:

1. **CURRENT_BASELINE:** raw diamond variables plus the original REG geometry.
2. **HUMAN_ONLY:** human-readable valuation features without raw x/y/z/depth/table.
3. **HUMAN_PLUS_RAW:** human features plus the five raw geometry measurements.

Every candidate uses the same rows. Models are selected by validation MAE. The test
set is evaluated once after final selection.

## 2. Configuration

Normal execution trains the full models. Set `DIAMOND_SMOKE_TEST=1` before starting
Jupyter for a quick structural run; smoke-test scores are not final results.

In [ ]:
from pathlib import Path
import json
import os
import sys
from urllib.request import Request, urlopen

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Find the Diamond project from either its root or a child folder.
PROJECT_ROOT = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "src" / "regression" / "training.py").is_file()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.regression.preprocessing import prepare_regression_data
from src.regression.datasets import export_datasets, show_dataframe, show_experiment_data
from src.regression.training import train_candidates, select_best_run, evaluate_on_test
from src.regression.evaluation import price_band_report
from src.regression.visualizations import plot_eda, plot_comparison, plot_top_features, plot_errors
from src.regression.prediction import save_best_model, load_best_model, predict_prices
from src.regression.api import start_prediction_api

RANDOM_STATE = 42
SMOKE_TEST = os.getenv("DIAMOND_SMOKE_TEST") == "1"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "diamonds.csv"
RUN_FOLDER = "smoke_test" if SMOKE_TEST else ""
DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "regression" / RUN_FOLDER
FIGURE_DIR = PROJECT_ROOT / "outputs" / "regression" / "figures" / RUN_FOLDER
REPORT_DIR = PROJECT_ROOT / "outputs" / "regression" / "reports" / RUN_FOLDER
MODEL_DIR = PROJECT_ROOT / "models" / "regression" / RUN_FOLDER
for folder in [DATASET_DIR, FIGURE_DIR, REPORT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Smoke test:", SMOKE_TEST)
print("Raw dataset:", DATA_PATH)

## 3. Load Raw Dataset

The maintained workflow starts from `data/raw/diamonds.csv`. The next call reproduces
the earlier regression population instead of silently using a historical export.

In [ ]:
prepared = prepare_regression_data(DATA_PATH, RANDOM_STATE)
show_dataframe(prepared["original"], "Original diamonds")

## 4. Data Cleaning

The workflow removes rows with nonpositive dimensions, then applies the original
carat-band volume rule: outside 70% of the band P10 through 130% of its P90.
This reproduces 53,899 rows. Duplicate observations are retained for parity with
the historical regression experiments.

In [ ]:
print("Original rows:", len(prepared["original"]))
print("Cleaned rows:", len(prepared["cleaned"]))
print("Removed rows:", len(prepared["removed"]))
show_dataframe(prepared["cleaned"], "Cleaned regression data")
display(prepared["removed"])

## 5. EDA and Findings

These plots describe the modeling population. Correlation is descriptive and does
not prove causation.

In [ ]:
plot_eda(prepared["featured"], FIGURE_DIR / "eda")

## 6. Base Feature Engineering

The original eight REG measurements describe size, proportions, asymmetry, and
weight relative to face area and volume.

In [ ]:
reg_columns = [column for column in prepared["featured"] if column.startswith("REG_")]
show_dataframe(prepared["featured"][["price"] + reg_columns], "Base REG features")

## 7. Train, Validation, and Test Split

The shared seed-42 split contains 37,729 training, 8,085 validation, and 8,085 test
rows. It is intentionally non-stratified because price is continuous.

In [ ]:
split_summary = pd.DataFrame([
    {"Split": split, "Rows": len(frame), "Minimum price": frame["price"].min(),
     "Median price": frame["price"].median(), "Maximum price": frame["price"].max()}
    for split, frame in prepared["raw"].items()
])
display(split_summary)

## 8. Human Feature Engineering

Polynomial face-area and volume expectations, peer medians, and rarity counts are
fitted on training rows only. Validation, test, and future API inputs reuse that
saved context, preventing target leakage.

In [ ]:
human_columns = [column for column in prepared["human"]["train"] if column.startswith("H_")]
print("Human features:", len(human_columns))
show_dataframe(prepared["human"]["train"][human_columns], "Training human features")

# Save all source and processed DataFrames in an organized regression folder.
dataset_files = export_datasets(prepared, prepared["cleaned"], DATASET_DIR)
display(dataset_files)

## 9. Experiment 1 — CURRENT_BASELINE

Original variables and REG features. XGBoost and Random Forest use the same encoded inputs and log-price
target. Validation MAE selects the stronger model for the top-eight plots.

In [ ]:
show_experiment_data(prepared, "CURRENT_BASELINE")
experiment_1_runs = train_candidates(
    prepared, ["CURRENT_BASELINE"], ["XGBoost", "RandomForest"], smoke=SMOKE_TEST
)
experiment_1_results = pd.DataFrame([run["metrics"] for run in experiment_1_runs])
display(experiment_1_results.sort_values("MAE"))
plot_comparison(experiment_1_results, FIGURE_DIR / "experiment_1", "validation_comparison")

experiment_1_best = select_best_run(experiment_1_runs)
importance_1 = plot_top_features(
    experiment_1_best,
    prepared["experiments"]["CURRENT_BASELINE"]["frames"]["valid"],
    prepared["targets"]["valid"],
    FIGURE_DIR / "experiment_1",
    "current_baseline",
    max_samples=250 if SMOKE_TEST else 1500,
    repeats=1 if SMOKE_TEST else 3,
)
display(importance_1.head(8))

## 10. Experiment 2 — HUMAN_ONLY

Human-readable valuation features without raw geometry. XGBoost and Random Forest use the same encoded inputs and log-price
target. Validation MAE selects the stronger model for the top-eight plots.

In [ ]:
show_experiment_data(prepared, "HUMAN_ONLY")
experiment_2_runs = train_candidates(
    prepared, ["HUMAN_ONLY"], ["XGBoost", "RandomForest"], smoke=SMOKE_TEST
)
experiment_2_results = pd.DataFrame([run["metrics"] for run in experiment_2_runs])
display(experiment_2_results.sort_values("MAE"))
plot_comparison(experiment_2_results, FIGURE_DIR / "experiment_2", "validation_comparison")

experiment_2_best = select_best_run(experiment_2_runs)
importance_2 = plot_top_features(
    experiment_2_best,
    prepared["experiments"]["HUMAN_ONLY"]["frames"]["valid"],
    prepared["targets"]["valid"],
    FIGURE_DIR / "experiment_2",
    "human_only",
    max_samples=250 if SMOKE_TEST else 1500,
    repeats=1 if SMOKE_TEST else 3,
)
display(importance_2.head(8))

## 11. Experiment 3 — HUMAN_PLUS_RAW

Human features with raw geometry restored. XGBoost and Random Forest use the same encoded inputs and log-price
target. Validation MAE selects the stronger model for the top-eight plots.

In [ ]:
show_experiment_data(prepared, "HUMAN_PLUS_RAW")
experiment_3_runs = train_candidates(
    prepared, ["HUMAN_PLUS_RAW"], ["XGBoost", "RandomForest"], smoke=SMOKE_TEST
)
experiment_3_results = pd.DataFrame([run["metrics"] for run in experiment_3_runs])
display(experiment_3_results.sort_values("MAE"))
plot_comparison(experiment_3_results, FIGURE_DIR / "experiment_3", "validation_comparison")

experiment_3_best = select_best_run(experiment_3_runs)
importance_3 = plot_top_features(
    experiment_3_best,
    prepared["experiments"]["HUMAN_PLUS_RAW"]["frames"]["valid"],
    prepared["targets"]["valid"],
    FIGURE_DIR / "experiment_3",
    "human_plus_raw",
    max_samples=250 if SMOKE_TEST else 1500,
    repeats=1 if SMOKE_TEST else 3,
)
display(importance_3.head(8))

## 12. Compare Feature Experiments

This table answers whether human features replace or complement raw geometry. The
winner is selected from freshly computed validation MAE; historical `$245.46` test
MAE is reference evidence only.

In [ ]:
feature_runs = experiment_1_runs + experiment_2_runs + experiment_3_runs
feature_results = pd.DataFrame([run["metrics"] for run in feature_runs]).sort_values("MAE")
display(feature_results)
feature_results.to_csv(REPORT_DIR / "feature_experiment_validation.csv", index=False)
plot_comparison(feature_results, FIGURE_DIR / "comparison", "feature_experiments")

## 13. Select the Winning Representation

The representation with the lowest validation MAE advances to the algorithm
comparison. This decision does not inspect test labels.

In [ ]:
best_feature_run = select_best_run(feature_runs)
winning_feature_set = best_feature_run["feature_set"]
print("Winning validation representation:", winning_feature_set)

## 14. Final Algorithm Comparison

XGBoost, Random Forest, CatBoost, LightGBM, and ExtraTrees are compared on the same
winning representation and the same training/validation rows.

In [ ]:
existing_family_runs = [run for run in feature_runs if run["feature_set"] == winning_feature_set]
additional_family_runs = train_candidates(
    prepared, [winning_feature_set], ["CatBoost", "LightGBM", "ExtraTrees"], smoke=SMOKE_TEST
)
family_runs = existing_family_runs + additional_family_runs
family_results = pd.DataFrame([run["metrics"] for run in family_runs]).sort_values("MAE")
display(family_results)
family_results.to_csv(REPORT_DIR / "algorithm_validation.csv", index=False)
plot_comparison(family_results, FIGURE_DIR / "comparison", "algorithm_validation")

## 15. Select the Final Model

The final winner is the lowest-validation-MAE candidate. If smoke mode is active,
the result proves wiring only and is marked as a smoke model.

In [ ]:
best_run = select_best_run(family_runs)
print("Selected algorithm:", best_run["algorithm"])
print("Selected representation:", best_run["feature_set"])
print("Validation MAE:", best_run["metrics"]["MAE"])

## 16. Detailed Held-Out Test Error Analysis

This is the first and only model-selection-stage use of test labels. The notebook
reports dollar errors, percentage errors, severity thresholds, and price bands.

In [ ]:
test_predictions, test_metrics = evaluate_on_test(best_run, prepared)
test_metrics_df = pd.DataFrame([test_metrics])
display(test_metrics_df.T)

band_metrics = price_band_report(prepared["targets"]["test"], test_predictions)
display(band_metrics)
test_metrics_df.to_csv(REPORT_DIR / "final_test_metrics.csv", index=False)
band_metrics.to_csv(REPORT_DIR / "final_price_band_metrics.csv", index=False)

test_results = prepared["raw"]["test"][["carat", "cut", "color", "clarity", "price"]].copy()
test_results["Predicted_Price"] = test_predictions
test_results["Absolute_Error"] = np.abs(test_results["price"] - test_predictions)
test_results["Percentage_Error"] = test_results["Absolute_Error"] / test_results["price"] * 100
test_results.to_csv(REPORT_DIR / "final_test_predictions.csv", index_label="Source_Row")
display(test_results)
plot_errors(prepared["targets"]["test"], test_predictions, band_metrics, FIGURE_DIR / "final")

## 17. Results and Business Interpretation

MAE describes average dollar error; median AE describes typical dollar error. MAPE
and the percentile bands make errors comparable across price levels. Results from a
smoke run must not be reported as final performance.

In [ ]:
if SMOKE_TEST:
    print("SMOKE TEST COMPLETE — do not use these small-model metrics as final results.")
else:
    print(f"The selected {best_run['algorithm']} / {best_run['feature_set']} model has")
    print(f"test MAE ${test_metrics['MAE']:,.2f}, median AE ${test_metrics['Median_AE']:,.2f},")
    print(f"MAPE {test_metrics['MAPE']:.2f}%, and R² {test_metrics['R2']:.4f}.")

## 18. Save Model, Predict New Prices, and Start the API

The saved bundle contains the fitted model, encoder, and training-only human context.
Supply nine raw inputs; omit price because price is the prediction target. The local
regression API uses port 8766 so it does not conflict with classification on 8765.

In [ ]:
save_best_model(
    best_run,
    MODEL_DIR,
    provenance={"data": str(DATA_PATH), "selected_by": "validation MAE", "test_evaluations": 1},
)
saved_run = load_best_model(MODEL_DIR)

example_diamond = {
    "carat": 0.70, "cut": "Ideal", "color": "G", "clarity": "VS1",
    "depth": 61.5, "table": 57.0, "x": 5.70, "y": 5.72, "z": 3.51,
}
display(pd.DataFrame(predict_prices(saved_run, example_diamond)))

In [ ]:
# Keep the kernel running while using this local endpoint.
if "regression_server" in globals():
    regression_server.shutdown()
    regression_server.server_close()
regression_server = start_prediction_api(MODEL_DIR, port=8766)

# Demonstrate a real request through the HTTP API.
request = Request(
    "http://127.0.0.1:8766/predict",
    data=json.dumps(example_diamond).encode("utf-8"),
    headers={"Content-Type": "application/json"},
    method="POST",
)
with urlopen(request, timeout=30) as response:
    display(json.loads(response.read()))